## Notebook conventions

**Edit this file in place — don't save a new copy** (no `_V1`/`_V2`/dated/`-Copy`/`DEBUG-` variants). Commit changes via a branch + PR; git history is the version record, not the filename. Full conventions and `nbstripout` setup: see the repo [README](../../README.md) → "Notebook conventions."

**Note:** adapted from `Notebooks/windows/BatchProcessing_with_filedialog.ipynb` for headless use on Altair (2026-08-29) — the three `tkinter` file-picker dialogs (input folder, output folder, model file) are replaced with hardcoded paths below, since `tk.Tk()` needs a real display this JupyterHub server doesn't have. Also added an optional timer cell and a "Real-Data Baseline Summary" cell at the end, matching the pattern used in `Batch-Video_Processing_2026.ipynb`.

# Batch Image Processing with MegaDetector

This notebook runs MegaDetector on a folder of camera trap images, then renders annotated output images (for visual review / Timelapse-style browsing) rather than just producing the raw detections JSON.

**If you are unsure what any of this does, see First_time_setup first.**

This version calls the **pip-installed** `megadetector` package (via `python -m ...`) using the exact Python interpreter behind the current kernel (`sys.executable`), instead of invoking script files from an old GitHub clone. This avoids two issues seen previously on Tarazed:

1. A stray machine-wide `PYTHONPATH` (now removed) that used to make old-clone-style invocations work by accident, while silently shadowing pip installs elsewhere.
2. Bare `!python ...` shell calls resolving to whichever Python happens to be first on the shell's PATH, which is not necessarily the environment behind the notebook's selected kernel.

Input/output folders and the model path are hardcoded below (see the note above) for headless use on this server, rather than chosen via interactive file-picker dialogs (that's still how the Windows version, `Video_Processing_Windows.ipynb`, does it).

## Setup

In [ ]:
import os
import sys
import subprocess
import time
from pathlib import Path

print("Imports successful.")

In [ ]:
# Sanity check: confirm this kernel is backed by the environment that has
# megadetector pip-installed (should point into the "megadetector" env).
print(sys.executable)

In [ ]:
# Confirm megadetector is actually installed in this environment.
# Note: megadetector.__version__ is not reliable/does not exist on the module -
# use importlib.metadata instead.
import importlib.metadata
print("megadetector version:", importlib.metadata.version("megadetector"))

In [ ]:
# Optional: start a timer so the summary cell at the end of this notebook can
# report elapsed wall-clock time. Safe to skip -- if you don't run this cell,
# the summary cell just skips the timing line instead of erroring.
RUN_START = time.perf_counter()
print(f"Timer started at {time.strftime('%H:%M:%S')}")

## Select folders

Hardcoded below rather than interactive file-picker dialogs — `tkinter` dialogs need a real display, which this headless JupyterHub server doesn't have (see the note near the top of this notebook). Edit `folder_path`, `RUN_LABEL`, and `model_path` below for a different run.

In [ ]:
# Input folder containing images to process.
# Hardcoded (no file-picker dialog -- see note above) for headless use on Altair.
# Set to the shared Siskiyou Pass Forest baseline dataset for the current baseline run.
folder_path = "/home/jupyter-bernie/Siskiyou_Pass_Forest_10Aug2026"
print("Input folder:", folder_path)

In [ ]:
# Output location -- auto-named per run (label + timestamp), so repeat runs
# never collide and never need a manual edit/commit just to bump a run number.
# Change RUN_LABEL to describe what this run is for (e.g. "after-fix").
RUN_LABEL = "baseline"
OUTPUT_DIR = Path(f"./megadetector_outputs/{RUN_LABEL}_{time.strftime('%Y%m%d_%H%M%S')}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

run_name = Path(folder_path).name
output_json = str(OUTPUT_DIR / f"{run_name}.json")
output_path = str(OUTPUT_DIR / "detections")

print("Output folder:", OUTPUT_DIR)
print("Detections JSON will be written to:", output_json)
print("Rendered images will be written to:", output_path)

In [ ]:
# MegaDetector model weights: a real path to the .pt file, not a short name
# like "MDV5A" -- a short name makes MegaDetector auto-download to ephemeral
# /tmp. Staged persistently instead at /shared/megadetector_models, the same
# shared, multi-user location the video notebook uses -- any user can read it.
model_path = "/shared/megadetector_models/md_v5a.0.1.pt"
print("Model file:", model_path)

In [ ]:
# Verify everything actually exists before running anything.
print("folder_path:", folder_path, "| exists:", os.path.isdir(folder_path))
print("model_path:", model_path, "| exists:", os.path.isfile(model_path))

assert os.path.isdir(folder_path), f"Input folder not found: {folder_path}"
assert os.path.isfile(model_path), f"Model file not found: {model_path}"

## Run detection

Runs MegaDetector over every image in `folder_path` and writes the raw results to `output_json`. `--checkpoint_frequency 1000` writes periodic checkpoints in case of a crash on a large batch.

In [ ]:
result = subprocess.run([
    sys.executable, "-m", "megadetector.detection.run_detector_batch",
    model_path,
    folder_path,
    output_json,
    "--output_relative_filenames",
    "--recursive",
    "--checkpoint_frequency", "1000",
    "--quiet",
], capture_output=True, text=True)

print("RETURN CODE:", result.returncode)
print("---STDOUT---")
print(result.stdout)
print("---STDERR---")
print(result.stderr)

assert result.returncode == 0, "Detection failed - see stderr above."

## Postprocess / render annotated images

Reads `output_json` and renders annotated copies of the images (with bounding boxes) plus an `index.html` viewer into `output_path`. Only run this after confirming the detection cell above finished with return code 0.

In [ ]:
result = subprocess.run([
    sys.executable, "-m", "megadetector.postprocessing.postprocess_batch_results",
    output_json,
    output_path,
    "--image_base_dir", folder_path,
    "--num_images_to_sample", "-1",
], capture_output=True, text=True)

print("RETURN CODE:", result.returncode)
print("---STDOUT---")
print(result.stdout)
print("---STDERR---")
print(result.stderr)

assert result.returncode == 0, "Postprocessing failed - see stderr above."

## Done

If both cells above returned code 0, open `output_path/index.html` in a browser to review the annotated images.

In [ ]:
print("Review results at:", os.path.join(output_path, "index.html"))

## Optional: Real-Data Baseline Summary

Run any time after the detection and postprocess cells above have finished, to get comparable before/after numbers for the metrics log.

In [ ]:
# Optional: real-data baseline summary for the paper/poster. Purely additive --
# doesn't change or depend on anything else in the notebook.
import json

# Same default MegaDetector's own postprocessing step uses to decide what
# counts as a real detection for the rendered detections_animal/non_detections
# folders -- matching it here so this means the same thing as those folders.
DETECTION_CONFIDENCE_THRESHOLD = 0.2

# MegaDetector's standard category IDs. Only "animal" matters for this
# project -- "person" hits are almost always just camera setup/check visits
# (confirmed by eye, 2026-08-29: field crew right up on the lens), not signal.
CATEGORY_ANIMAL = "1"
CATEGORY_PERSON = "2"
CATEGORY_VEHICLE = "3"

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
input_images = [p for p in Path(folder_path).rglob("*") if p.suffix.lower() in IMAGE_EXTS]

with open(output_json) as f:
    md = json.load(f)
n_results = len(md.get("images", []))

def images_with_category(category_id):
    return sum(
        1 for img in md.get("images", [])
        if any(
            d.get("category") == category_id and d.get("conf", 0) >= DETECTION_CONFIDENCE_THRESHOLD
            for d in img.get("detections", [])
        )
    )

n_animal = images_with_category(CATEGORY_ANIMAL)
n_person = images_with_category(CATEGORY_PERSON)
n_vehicle = images_with_category(CATEGORY_VEHICLE)

print("=" * 55)
print("REAL-DATA BASELINE SUMMARY")
print("=" * 55)
print(f"Input images found                              : {len(input_images)}")
print(f"Images with results in detections.json          : {n_results}")
print(f"Images with a real ANIMAL detection (conf >= {DETECTION_CONFIDENCE_THRESHOLD}) : {n_animal}  <-- the number that matters")
print(f"  (person detections, not counted above)        : {n_person}")
print(f"  (vehicle detections, not counted above)       : {n_vehicle}")
if "RUN_START" in globals():
    elapsed = time.perf_counter() - RUN_START
    print(f"Elapsed wall-clock time (detection + postprocess): {elapsed:.1f}s ({elapsed/60:.1f} min)")
print(f"Output folder: {OUTPUT_DIR}")
print("=" * 55)